# Gemma 4 Local Inference — Cross-Platform Setup & Viability Notebook

> **Run Google's Gemma 4 models on your own machine.** This notebook detects your hardware (macOS or Windows), recommends viable models, installs the right inference engine, and benchmarks throughput.

**Supports:**
- macOS Apple Silicon (M1/M2/M3/M4) via **MLX** (optimal)
- Windows/Linux with NVIDIA GPU via **llama-cpp-python + CUDA**
- Windows/Linux/Mac Intel CPU-only via **llama-cpp-python** (slower)

---

| Step | What | Time |
|------|------|------|
| 1 | Hardware detection (OS, CPU, RAM, GPU) | ~2s |
| 2 | Model viability assessment | instant |
| 3 | Install inference engine | 1-10 min |
| 4 | Download model | 1-20 min |
| 5 | Benchmark inference (tok/s) | ~1 min |
| 6 | (Optional) Local gateway setup | manual |
| 7 | (Optional) GCP Cloud Run GPU | manual |

---
## Step 1: Hardware Detection

Identifies your OS, CPU, total/available RAM, GPU type and VRAM, and free disk space.

In [ ]:
import platform, os, sys, subprocess, shutil, json
from pathlib import Path

# ── OS Detection ────────────────────────────────────────────────
OS_NAME = platform.system()
OS_LABEL = {'Darwin': 'macOS', 'Windows': 'Windows', 'Linux': 'Linux'}.get(OS_NAME, OS_NAME)
ARCH = platform.machine()
IS_MAC = OS_NAME == 'Darwin'
IS_WINDOWS = OS_NAME == 'Windows'
IS_LINUX = OS_NAME == 'Linux'
IS_APPLE_SILICON = IS_MAC and ARCH == 'arm64'

# ── CPU ──────────────────────────────────────────────────────────
CPU_NAME = platform.processor() or 'Unknown'
if IS_MAC:
    try:
        CPU_NAME = subprocess.check_output(
            ['sysctl', '-n', 'machdep.cpu.brand_string'], text=True,
            stderr=subprocess.DEVNULL
        ).strip()
    except Exception:
        try:
            CPU_NAME = subprocess.check_output(
                ['sysctl', '-n', 'hw.chip'], text=True,
                stderr=subprocess.DEVNULL
            ).strip()
        except Exception:
            pass
elif IS_WINDOWS:
    try:
        lines = subprocess.check_output(
            ['wmic', 'cpu', 'get', 'name'], text=True,
            stderr=subprocess.DEVNULL
        ).strip().split('\n')
        CPU_NAME = [l.strip() for l in lines if l.strip() and l.strip() != 'Name'][-1]
    except Exception:
        pass
elif IS_LINUX:
    try:
        with open('/proc/cpuinfo') as f:
            for line in f:
                if 'model name' in line:
                    CPU_NAME = line.split(':')[1].strip()
                    break
    except Exception:
        pass

CPU_CORES = os.cpu_count() or 0

# ── Memory ───────────────────────────────────────────────────────
def get_memory_gb():
    try:
        import psutil
        mem = psutil.virtual_memory()
        return round(mem.total / 1e9, 1), round(mem.available / 1e9, 1)
    except ImportError:
        pass
    if IS_MAC:
        try:
            total = int(subprocess.check_output(
                ['sysctl', '-n', 'hw.memsize'], text=True
            ).strip())
            return round(total / 1e9, 1), round(total * 0.6 / 1e9, 1)
        except Exception:
            pass
    elif IS_WINDOWS:
        try:
            out = subprocess.check_output(
                ['wmic', 'OS', 'get', 'TotalVisibleMemorySize,FreePhysicalMemory'],
                text=True
            )
            lines = [l.strip() for l in out.strip().split('\n') if l.strip()]
            if len(lines) >= 2:
                parts = lines[-1].split()
                return round(int(parts[-1]) / 1e6, 1), round(int(parts[0]) / 1e6, 1)
        except Exception:
            pass
    elif IS_LINUX:
        try:
            info = {}
            with open('/proc/meminfo') as f:
                for line in f:
                    parts = line.split(':')
                    if len(parts) == 2:
                        info[parts[0].strip()] = int(parts[1].strip().split()[0])
            total = info.get('MemTotal', 0) / 1e6
            avail = info.get('MemAvailable', info.get('MemFree', 0)) / 1e6
            return round(total, 1), round(avail, 1)
        except Exception:
            pass
    return 0.0, 0.0

TOTAL_RAM_GB, AVAIL_RAM_GB = get_memory_gb()

# ── GPU Detection ────────────────────────────────────────────────
GPU_TYPE = 'none'
GPU_NAME = 'None detected'
GPU_VRAM_GB = 0.0

if IS_APPLE_SILICON:
    GPU_TYPE = 'apple_silicon'
    GPU_NAME = f'{CPU_NAME} (unified memory)'
    GPU_VRAM_GB = TOTAL_RAM_GB
else:
    try:
        nv = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name,memory.total',
             '--format=csv,noheader,nounits'],
            text=True, stderr=subprocess.DEVNULL
        ).strip()
        if nv:
            parts = nv.split(',')
            GPU_TYPE = 'nvidia'
            GPU_NAME = parts[0].strip()
            GPU_VRAM_GB = round(float(parts[1].strip()) / 1024, 1)
    except (FileNotFoundError, subprocess.CalledProcessError):
        pass

# ── Disk ──────────────────────────────────────────────────────────
disk_usage = shutil.disk_usage(Path.home())
DISK_FREE_GB = round(disk_usage.free / 1e9, 1)

# ── Report ────────────────────────────────────────────────────────
print('=' * 60)
print('  HARDWARE DETECTION REPORT')
print('=' * 60)
print(f'  OS:              {OS_LABEL} ({ARCH})')
print(f'  CPU:             {CPU_NAME}')
print(f'  CPU Cores:       {CPU_CORES}')
print(f'  RAM Total:       {TOTAL_RAM_GB} GB')
print(f'  RAM Available:   {AVAIL_RAM_GB} GB')
print(f'  GPU:             {GPU_NAME}')
print(f'  GPU Type:        {GPU_TYPE}')
print(f'  GPU VRAM:        {GPU_VRAM_GB} GB'
      + (' (shared)' if GPU_TYPE == 'apple_silicon' else ''))
print(f'  Disk Free:       {DISK_FREE_GB} GB')
print(f'  Apple Silicon:   {"Yes" if IS_APPLE_SILICON else "No"}')
print('=' * 60)

---
## Step 2: Model Viability Assessment

Checks which Gemma 4 models your hardware can run, and selects the best inference engine.

In [ ]:
MODELS = [
    {
        'name': 'Gemma 4 E2B (2.3B active)',
        'tier': 'fast',
        'mlx_id': 'mlx-community/gemma-4-e2b-it-4bit',
        'gguf_repo': 'unsloth/gemma-4-E2B-it-GGUF',
        'gguf_file': 'gemma-4-E2B-it-UD-Q4_K_XL.gguf',
        'ram_gb': 4.0, 'vram_gb': 2.0, 'disk_gb': 1.5,
        'tps_mlx': '100-130', 'tps_cuda': '80-120', 'tps_cpu': '15-25',
        'use': 'Real-time chat, simple Q&A, triage',
    },
    {
        'name': 'Gemma 4 E4B (4.5B active)',
        'tier': 'primary',
        'mlx_id': 'mlx-community/gemma-4-e4b-it-4bit',
        'gguf_repo': 'unsloth/gemma-4-E4B-it-GGUF',
        'gguf_file': 'gemma-4-E4B-it-UD-Q4_K_XL.gguf',
        'ram_gb': 6.0, 'vram_gb': 3.5, 'disk_gb': 3.0,
        'tps_mlx': '28-35', 'tps_cuda': '50-80', 'tps_cpu': '5-10',
        'use': 'Code generation, summarization, reasoning',
    },
    {
        'name': 'Gemma 4 26B-A4B (3.8B active, MoE)',
        'tier': 'heavy',
        'mlx_id': 'mlx-community/gemma-4-26b-a4b-it-4bit',
        'gguf_repo': 'unsloth/gemma-4-26B-A4B-it-GGUF',
        'gguf_file': 'gemma-4-26B-A4B-it-UD-Q4_K_XL.gguf',
        'ram_gb': 20.0, 'vram_gb': 18.0, 'disk_gb': 18.0,
        'tps_mlx': '50-70 (36+ GB RAM)', 'tps_cuda': '50-70 (L4/4090)', 'tps_cpu': '1-3',
        'use': 'Complex analysis, long context (256K), deep code gen',
    },
]

# Select engine
if IS_APPLE_SILICON:
    ENGINE = 'mlx'
    ENGINE_LABEL = 'MLX (Apple Silicon — optimal)'
elif GPU_TYPE == 'nvidia':
    ENGINE = 'llama_cpp_cuda'
    ENGINE_LABEL = f'llama-cpp-python + CUDA ({GPU_NAME})'
else:
    ENGINE = 'llama_cpp_cpu'
    ENGINE_LABEL = 'llama-cpp-python (CPU-only — slower)'

print('=' * 60)
print(f'  ENGINE: {ENGINE_LABEL}')
print('=' * 60 + '\n')

viable_models = []

for m in MODELS:
    if ENGINE == 'mlx':
        mem_ok = TOTAL_RAM_GB >= m['ram_gb']
        mem_need = f"{m['ram_gb']} GB unified RAM"
        speed = m['tps_mlx']
    elif ENGINE == 'llama_cpp_cuda':
        mem_ok = GPU_VRAM_GB >= m['vram_gb']
        mem_need = f"{m['vram_gb']} GB VRAM"
        speed = m['tps_cuda']
    else:
        mem_ok = TOTAL_RAM_GB >= m['ram_gb']
        mem_need = f"{m['ram_gb']} GB RAM"
        speed = m['tps_cpu']
    
    disk_ok = DISK_FREE_GB >= m['disk_gb']
    can_run = mem_ok and disk_ok
    icon = '\u2705' if can_run else '\u274c'
    
    print(f"  {icon}  {m['name']}")
    print(f"       Needs: {mem_need} + {m['disk_gb']} GB disk")
    print(f"       Speed: ~{speed} tok/s")
    print(f"       Use:   {m['use']}")
    if not mem_ok:
        have = GPU_VRAM_GB if ENGINE == 'llama_cpp_cuda' else TOTAL_RAM_GB
        print(f"       \u26a0  Insufficient memory ({have} GB available)")
    if not disk_ok:
        print(f"       \u26a0  Insufficient disk ({DISK_FREE_GB} GB free)")
    print()
    if can_run:
        viable_models.append(m)

if viable_models:
    SELECTED = viable_models[0]
    print(f'Selected for setup: {SELECTED["name"]} ({SELECTED["tier"]} tier)')
else:
    SELECTED = None
    print('\u26a0  No models viable locally. See Step 7 for GCP Cloud Run GPU.')

---
## Step 3: Install Inference Engine

Installs MLX (Apple Silicon) or llama-cpp-python (NVIDIA/CPU) based on detection.

In [ ]:
def pip_install(pkgs, extra=None):
    cmd = [sys.executable, '-m', 'pip', 'install', '--upgrade'] + (extra or []) + pkgs
    print(f"$ {' '.join(cmd)}\n")
    r = subprocess.run(cmd)
    print('\u2705 Done\n' if r.returncode == 0 else f'\u274c Failed (exit {r.returncode})\n')
    return r.returncode == 0

if ENGINE == 'mlx':
    print('Installing MLX for Apple Silicon...\n')
    print('NOTE: Gemma 4 needs mlx-lm >= 0.31.2. If PyPI is behind,\n'
          '      install from GitHub: pip install git+https://github.com/'
          'ml-explore/mlx-examples.git#subdirectory=llms/mlx_lm\n')
    ok = pip_install(['mlx', 'mlx-lm>=0.31.2'])
    if not ok:
        print('Trying GitHub main...')
        pip_install(['git+https://github.com/ml-explore/mlx-examples.git'
                     '#subdirectory=llms/mlx_lm'])

elif ENGINE == 'llama_cpp_cuda':
    print('Installing llama-cpp-python with CUDA...\n')
    print('This compiles llama.cpp with CUDA — may take 5-10 minutes.\n')
    os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=ON'
    os.environ['FORCE_CMAKE'] = '1'
    pip_install(['llama-cpp-python>=0.2.80'], extra=['--no-cache-dir'])

else:
    print('Installing llama-cpp-python (CPU-only)...\n')
    print('\u26a0  CPU is significantly slower. Consider the Cloud Run option (Step 7).\n')
    pip_install(['llama-cpp-python>=0.2.80'])

pip_install(['huggingface_hub[hf_transfer]', 'fastapi', 'uvicorn[standard]'])

---
## Step 4: Download Model

Downloads the recommended model. MLX models from `mlx-community`, GGUF from `unsloth` (Unsloth Dynamic turboquant).

In [ ]:
import time

if SELECTED is None:
    print('\u26a0  No viable models. Skip to Step 7 for cloud setup.')
    MODEL_PATH = None
else:
    os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
    
    if ENGINE == 'mlx':
        repo = SELECTED['mlx_id']
        print(f'Downloading MLX model: {repo}')
        print(f'Estimated size: ~{SELECTED["disk_gb"]} GB\n')
        from huggingface_hub import snapshot_download
        t0 = time.time()
        MODEL_PATH = snapshot_download(
            repo_id=repo,
            allow_patterns=['*.safetensors', '*.json', 'tokenizer*'],
        )
        print(f'\n\u2705 Downloaded to: {MODEL_PATH}')
        print(f'   Time: {time.time() - t0:.0f}s')
    else:
        repo = SELECTED['gguf_repo']
        fname = SELECTED['gguf_file']
        print(f'Downloading GGUF: {repo}/{fname}')
        print(f'Estimated size: ~{SELECTED["disk_gb"]} GB\n')
        from huggingface_hub import hf_hub_download
        t0 = time.time()
        MODEL_PATH = hf_hub_download(repo_id=repo, filename=fname)
        print(f'\n\u2705 Downloaded to: {MODEL_PATH}')
        print(f'   Time: {time.time() - t0:.0f}s')

---
## Step 5: Inference Benchmark

Measures tokens/second on your hardware with a standard prompt.

In [ ]:
PROMPT = ("Explain the key differences between transformer and recurrent "
          "neural network architectures in 3 concise paragraphs.")
MAX_TOKENS = 256

if MODEL_PATH is None:
    print('\u26a0  No model available. Skipping benchmark.')

elif ENGINE == 'mlx':
    print(f'Benchmarking MLX: {SELECTED["name"]}')
    print(f'Max tokens: {MAX_TOKENS}\n')
    from mlx_lm import load, generate
    
    t0 = time.time()
    model_obj, tokenizer = load(MODEL_PATH)
    load_s = time.time() - t0
    print(f'Model loaded in {load_s:.1f}s\n')
    
    # Warm-up
    _ = generate(model_obj, tokenizer, prompt='Hello', max_tokens=10, verbose=False)
    
    # Benchmark
    t0 = time.time()
    output = generate(model_obj, tokenizer, prompt=PROMPT,
                      max_tokens=MAX_TOKENS, verbose=True)
    elapsed = time.time() - t0
    out_tok = len(tokenizer.encode(output))
    tps = out_tok / elapsed if elapsed > 0 else 0
    
    print(f'\n{"=" * 50}')
    print(f'  Model:      {SELECTED["name"]}')
    print(f'  Engine:     MLX (Metal GPU)')
    print(f'  Tokens:     {out_tok}')
    print(f'  Time:       {elapsed:.2f}s')
    print(f'  Throughput: {tps:.1f} tok/s')
    print(f'  Load time:  {load_s:.1f}s')
    print(f'{"=" * 50}')
    del model_obj, tokenizer

else:
    print(f'Benchmarking llama.cpp: {SELECTED["name"]}')
    print(f'Backend: {"CUDA" if ENGINE == "llama_cpp_cuda" else "CPU"}')
    print(f'Max tokens: {MAX_TOKENS}\n')
    from llama_cpp import Llama
    
    n_gpu = -1 if ENGINE == 'llama_cpp_cuda' else 0
    t0 = time.time()
    llm = Llama(model_path=MODEL_PATH, n_ctx=4096,
                n_gpu_layers=n_gpu, n_threads=max(CPU_CORES - 2, 1),
                verbose=False)
    load_s = time.time() - t0
    print(f'Model loaded in {load_s:.1f}s\n')
    
    # Warm-up
    llm.create_chat_completion(
        messages=[{'role': 'user', 'content': 'Hi'}], max_tokens=10)
    
    # Benchmark
    t0 = time.time()
    result = llm.create_chat_completion(
        messages=[{'role': 'user', 'content': PROMPT}],
        max_tokens=MAX_TOKENS)
    elapsed = time.time() - t0
    out_tok = result.get('usage', {}).get('completion_tokens', 0)
    tps = out_tok / elapsed if elapsed > 0 else 0
    text = result['choices'][0]['message']['content']
    
    print(f'Response preview: {text[:200]}...\n')
    print(f'{"=" * 50}')
    print(f'  Model:      {SELECTED["name"]}')
    print(f'  Engine:     llama.cpp ({"CUDA" if ENGINE == "llama_cpp_cuda" else "CPU"})')
    print(f'  Tokens:     {out_tok}')
    print(f'  Time:       {elapsed:.2f}s')
    print(f'  Throughput: {tps:.1f} tok/s')
    print(f'  Load time:  {load_s:.1f}s')
    print(f'{"=" * 50}')
    del llm

---
## Step 6: (Optional) Local Gateway Setup

Instructions for running the full multi-tier FastAPI gateway with OpenAI-compatible API.

In [ ]:
print('LOCAL GATEWAY SETUP')
print('=' * 60 + '\n')

if IS_APPLE_SILICON:
    print('# === macOS Apple Silicon Setup ===\n')
    print('# Terminal 1: Fast tier (E2B, ~126 tok/s)')
    print('python -m mlx_lm.server \\\')
    print('  --model mlx-community/gemma-4-e2b-it-4bit \\\')
    print('  --port 8082 --host 0.0.0.0\n')
    print('# Terminal 2: Primary tier (E4B, ~32 tok/s)')
    print('python -m mlx_lm.server \\\')
    print('  --model mlx-community/gemma-4-e4b-it-4bit \\\')
    print('  --port 8083 --host 0.0.0.0\n')
    print('# Terminal 3: Gateway router')
    print('python scripts/gateway.py  # port 8080\n')
else:
    print('# === Windows / Linux Setup ===\n')
    if viable_models:
        m = SELECTED
        gpu_flag = '-1' if ENGINE == 'llama_cpp_cuda' else '0'
        print(f'# Start {m["name"]} server:')
        print(f'python -m llama_cpp.server \\\')
        print(f'  --model "{MODEL_PATH}" \\\')
        print(f'  --n_gpu_layers {gpu_flag} \\\')
        print(f'  --host 0.0.0.0 --port 8080\n')
    else:
        print('No local models available. Use GCP Cloud Run (Step 7).\n')

print('# Test with curl:')
print('curl -X POST http://localhost:8080/v1/chat/completions \\\')
print('  -H "Content-Type: application/json" \\\')
print('  -d \'{"messages":[{"role":"user","content":"Hello!"}]}\'\n')
print('# Or with Python OpenAI SDK:')
print('from openai import OpenAI')
print('client = OpenAI(base_url="http://localhost:8080/v1", api_key="none")')
print('r = client.chat.completions.create(')
print('    model="auto", messages=[{"role": "user", "content": "Hello!"}])')

---
## Step 7: (Optional) GCP Cloud Run GPU — Heavy Tier

If your hardware can't run the 26B model (needs 24+ GB VRAM), deploy it on Google Cloud Run with an NVIDIA L4 GPU. **Scale-to-zero = $0 when idle, ~$0.23/hr when active.**

### Prerequisites
- Google Cloud account with billing enabled
- `gcloud` CLI installed and authenticated
- HuggingFace account with [Gemma license accepted](https://huggingface.co/google/gemma-4-26B-A4B-it)

In [ ]:
print('GCP CLOUD RUN GPU DEPLOYMENT')
print('=' * 60 + '\n')
print('Deploys Gemma 4 26B on NVIDIA L4 GPU via Cloud Run.')
print('Cost: ~$0.23/hr active, $0 idle (scale-to-zero).\n')
print('-' * 60 + '\n')

steps = [
    ('Set credentials',
     'export PROJECT_ID="your-gcp-project"\n'
     'export HF_TOKEN="hf_..."\n'
     'export HEAVY_API_KEY=$(openssl rand -hex 32)'),
    ('Clone project',
     'git clone https://github.com/jemma/gemma4-stack.git\n'
     'cd gemma4-stack'),
    ('Deploy (automated script)',
     'bash cloud/heavy/deploy.sh $PROJECT_ID\n\n'
     '# Handles: APIs, Artifact Registry, GCS, Secrets,\n'
     '# Docker build (CUDA 12.4, ~10 min), Cloud Run deploy'),
    ('Test',
     'HEAVY_URL=$(gcloud run services describe gemma4-heavy \\\n'
     '  --region=us-central1 --format="value(status.url)")\n\n'
     'curl -X POST $HEAVY_URL/v1/chat/completions \\\n'
     '  -H "Content-Type: application/json" \\\n'
     '  -H "Authorization: Bearer $HEAVY_API_KEY" \\\n'
     '  -d \'{"messages":[{"role":"user","content":"Hello!"}]}\''),
    ('Tear down (stop billing)',
     'gcloud run services delete gemma4-heavy --region=us-central1 -q\n'
     'gcloud run services delete gemma4-proxy --region=us-central1 -q\n'
     'gcloud storage rm -r gs://$PROJECT_ID-models/ -q\n'
     'gcloud artifacts repositories delete gemma4 '
     '--location=us-central1 -q'),
]

for i, (title, cmd) in enumerate(steps, 1):
    print(f'# Step {i}: {title}')
    for line in cmd.split('\n'):
        print(f'  {line}')
    print()

---
## Summary

In [ ]:
print('\n' + '=' * 60)
print('  SESSION SUMMARY')
print('=' * 60)
print(f'  System:        {OS_LABEL} {ARCH} / {CPU_NAME}')
print(f'  RAM:           {TOTAL_RAM_GB} GB total, {AVAIL_RAM_GB} GB available')
print(f'  GPU:           {GPU_NAME} ({GPU_VRAM_GB} GB)')
print(f'  Engine:        {ENGINE_LABEL}')
print(f'  Models viable: {len(viable_models)} of {len(MODELS)}')
for m in viable_models:
    print(f'    \u2705 {m["name"]} ({m["tier"]})')
for m in MODELS:
    if m not in viable_models:
        kind = 'VRAM' if ENGINE == 'llama_cpp_cuda' else 'RAM'
        print(f'    \u274c {m["name"]} (needs more {kind})')
print()
if viable_models:
    print('  \u2192 Local Gemma 4 inference is viable on this machine.')
    print('    See Step 6 for gateway setup.')
else:
    print('  \u2192 Local inference not viable. Use GCP Cloud Run GPU (Step 7).')
print()
print('Full architecture study: see README.md')
print('=' * 60)